# 10 — Run the Stage 1 GHI pipeline (orchestrator)

Rebuilds Hossein's Stage 1 tables **cleanly**, writing to `*_v2` tables so his
originals are never touched. Run the cells **in order**, checking the output of
each before moving on.

```
build_structured_data  →  build_split_days  →  build_ghi_model  →  build_all_uncurtailedpv
        (_v2)                   (_v2)                (_v2)                  (_v2)
```

**Fixes baked into this pipeline** (from the review tally):
- **R1** structured_data uses `max(voltage)` not `avg(voltage)`
- **R2** structured_data excludes `flex_export_detected = True` sites
- **R6** all_uncurtailedpv caps the counterfactual at nameplate capacity

**Cost note:** the structured_data build scans `ts` (billions of rows). Run the
**tiny test slice first** (one month, one site-part) and confirm it works before
scaling to the full year. Athena bills by data scanned.

In [ ]:
# ── Bootstrap: paths + imports ──────────────────────────────────────────────
# EDIT the two paths: one to your shared folder, one to this stage1 folder.
import sys, pathlib
SHARED  = pathlib.Path(r"C:/Users/z3553082/.../ciccada_analysis/shared")              # <-- EDIT
STAGE1  = pathlib.Path(r"C:/Users/z3553082/.../ciccada_analysis/data_calc_write/stage1_ghi_pipeline")  # <-- EDIT
sys.path.insert(0, str(SHARED))
sys.path.insert(0, str(STAGE1))

from aws_config import aq            # your existing Athena helper
from ciccada_config import SAI       # 'solar_analytics_iceberg'

import build_structured_data   as b1
import build_split_days        as b2
import build_ghi_model         as b3
import build_all_uncurtailedpv as b4

print("Target tables (note the _v2 suffix — originals untouched):")
print(" ", b1.TARGET)
print(" ", b2.TARGET)
print(" ", b3.TARGET)
print(" ", b4.TARGET)

## Step 1 — structured_data_v2

In [ ]:
# 1a. Create the empty table (safe: drops & recreates only the _v2 table)
print(b1.create_table(aq, database=SAI))

In [ ]:
# 1b. TEST SLICE FIRST — one month, one of 8 site-parts. Cheap.
#     Confirm this completes and validate() looks sane BEFORE the full run.
b1.run_slice(aq, database=SAI, year=2024, months=[1], n_parts=8, parts=[0])

In [ ]:
# 1c. Validate the test slice
b1.validate(aq, database=SAI)

In [ ]:
# 1d. FULL RUN — all months, all 8 site-parts, for the year(s) you need.
#     Only run this once the test slice looks right. This is the expensive one.
#     (Re-running create_table first would wipe the test slice — that's expected;
#      the full run below reloads everything cleanly.)
print(b1.create_table(aq, database=SAI))
b1.run_slice(aq, database=SAI, year=2024, months=range(1, 13), n_parts=8)
# If you also have 2025 telemetry with GHI coverage, repeat for year=2025.
b1.validate(aq, database=SAI)

## Step 2 — split_days_v2

In [ ]:
print(b2.create_table(aq, database=SAI))
print(b2.run(aq, database=SAI))
b2.validate(aq, database=SAI)

## Step 3 — pv_ghi_norm_model_v2

In [ ]:
print(b3.create_table(aq, database=SAI))
b3.run_year(aq, database=SAI, year=2024)   # one shot; add year=2025 if present
b3.validate(aq, database=SAI)

## Step 4 — all_uncurtailedpv_v2

Needs your local `mape<50_sites.csv` quality-gate file. Point `MAPE_CSV` at it.
The `capped` column and `validate()` report how often the R6 nameplate cap bit —
quote that number in the paper.

In [ ]:
MAPE_CSV = r"C:/Users/z3553082/.../mape<50_sites.csv"   # <-- EDIT

print(b4.create_table(aq, database=SAI))
b4.run_year(aq, database=SAI, year=2024, mape_csv_path=MAPE_CSV, n_parts=3)
b4.validate(aq, database=SAI)

## Done — Stage 1 rebuilt

You now have `structured_data_v2`, `split_days_v2`, `pv_ghi_norm_model_v2`, and
`all_uncurtailedpv_v2`.

**To point notebook 03 at the clean tables**, change the table names in
`voltvar_queries.py` from `structured_data` → `structured_data_v2` and
`all_uncurtailedpv` → `all_uncurtailedpv_v2` (see the change-list I gave you).

**Next:** once you're happy with Stage 1, we build Stage 2 (the conformance
table builders) the same way.